# Proposed validation — review before it runs

**What this measures:** The target `explained_variance_gap__topk_minus_abstopk` is the difference between the same-seed TopK control's and the new AbsTopK's held-out `reconstruction_quality.explained_variance` from the repo's own `run_evals` on identical `tiny-stories-1M` activations (same k=100, d_sae=1024, seed, 2,048,000 tokens, CPU), so it directly answers "without hurting reconstruction" within the user's 0.02 band, while the guardrail `abstopk_negative_latent_share` (share of selected latents that are negative, counted `!= 0` / `< 0`) answers "recovers negative signals". Both trainings live inside ONE run as `question.kind: capability` with `baseline: none`, because `main` cannot import `AbsTopKTrainingSAE` at all and printing a stand-in number for it would be fabricated evidence; the TopK arm is the measured, in-run control, and the script additionally prints the repo-evals L0 (`> 0`) next to the true nonzero count per token to expose evals.py's definition gap on signed activations.

**Target metric:** `explained_variance_gap__topk_minus_abstopk`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

This notebook is the only copy of the test: `.remyx/validation.yaml` points here, and the run executes these cells top to bottom.

In [1]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

In [2]:
# Parameters
variant = "feature"
ref = "e496a76845b2bcf129393ce74d371809ab6c4a9f"
seed = 0


### Weights & Biases

This validation logs training runs to **Weights & Biases** so the dashboards the repository asks for exist. It needs your W&B API key (W&B → Settings → API keys). In Colab, store it as a secret named `WANDB_API_KEY`; elsewhere set the environment variable. The run goes to entity `smellslikeml` (your W&B username or a team you belong to) and project `remyx-validate-saelens` (the folder the runs are grouped under, created if it does not exist) — the same place the Remyx run logs to, so your run appears beside it. Change the two values below to log somewhere else.

In [3]:
import os
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception:
        pass
os.environ.setdefault("WANDB_ENTITY", "smellslikeml")
os.environ.setdefault("WANDB_PROJECT", "remyx-validate-saelens")
os.environ.setdefault("WANDB_BASE_URL", "https://api.wandb.ai")
os.environ.setdefault("WANDB_TAGS", "remyx-validate,hand-run")
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY not set — W&B logging disabled for this run")
else:
    print(f"W&B: {os.environ['WANDB_ENTITY']}/{os.environ['WANDB_PROJECT']}")

W&B: smellslikeml/remyx-validate-saelens


## Execution context

The cells below are the test Remyx drafted, split into cells; it was written as a script, so `__file__` is set to `eval/eval_abstopk_vs_topk.py` (no such file is committed). This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [4]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_abstopk_vs_topk.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

[remyx] cwd /workspace/target_repo | script /workspace/target_repo/eval/eval_abstopk_vs_topk.py


In [5]:
"""Paired TopK vs AbsTopK SAE training eval on tiny-stories-1M activations (CPU).

Trains one TopKTrainingSAE (control) and one AbsTopKTrainingSAE (test) with the
same k, width, seed and tokens through LanguageModelSAERunnerConfig +
LanguageModelSAETrainingRunner (logger=LoggingConfig(log_to_wandb=True)), then
reads held-out run_evals reconstruction_quality.explained_variance and sparsity
L0 (which counts acts > 0), plus sign-sensitive stats on held-out store batches
counted with != 0 -- the contrast the PR's docs snippet calls for.
"""

"Paired TopK vs AbsTopK SAE training eval on tiny-stories-1M activations (CPU).\n\nTrains one TopKTrainingSAE (control) and one AbsTopKTrainingSAE (test) with the\nsame k, width, seed and tokens through LanguageModelSAERunnerConfig +\nLanguageModelSAETrainingRunner (logger=LoggingConfig(log_to_wandb=True)), then\nreads held-out run_evals reconstruction_quality.explained_variance and sparsity\nL0 (which counts acts > 0), plus sign-sensitive stats on held-out store batches\ncounted with != 0 -- the contrast the PR's docs snippet calls for.\n"

In [6]:
import argparse
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

In [7]:
import torch

from sae_lens import (
    LanguageModelSAERunnerConfig,
    LanguageModelSAETrainingRunner,
    LoggingConfig,
    TopKTrainingSAEConfig,
    run_evals,
)
from sae_lens.evals import EvalConfig
from sae_lens.training.activation_scaler import ActivationScaler

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
try:  # exactly the symbols this PR adds; absent on the baseline
    from sae_lens import AbsTopKTrainingSAEConfig

    ABSTOPK_AVAILABLE = True
except ImportError:
    AbsTopKTrainingSAEConfig = None
    ABSTOPK_AVAILABLE = False

In [9]:
SEED = 42
K = 100  # repo TopK/AbsTopK default
D_IN = 64  # tiny-stories-1M d_model; blocks.1.hook_mlp_out dim
D_SAE = 1024  # 16x d_in, the repo's documented TopK width factor
os.environ.setdefault("WANDB_RUN_GROUP", "abstopk-vs-topk-tinystories")  # control + test in one W&B group

def build_cfg(sae_cfg, training_tokens):
    return LanguageModelSAERunnerConfig(
        model_name="tiny-stories-1M",
        dataset_path="roneneldan/TinyStories",
        hook_name="blocks.1.hook_mlp_out",
        training_tokens=training_tokens,
        n_batches_in_buffer=64,  # buffer = 64 x 128-token contexts = 8192 >= the 4096-token train batch (the store refuses smaller buffers)
        device="cpu",
        logger=LoggingConfig(  # entity/project/group come from the run environment
            log_to_wandb=os.environ.get("WANDB_MODE") != "disabled",
            wandb_entity=os.environ.get("WANDB_ENTITY"),
            wandb_project=os.environ.get("WANDB_PROJECT", "remyx-validate-saelens"),
        ),
        sae=sae_cfg,
    )

In [10]:
def run_arm(sae_cfg, training_tokens, n_eval_batches, sign_batches):
    torch.manual_seed(SEED)  # identical init + data order across both arms
    runner = LanguageModelSAETrainingRunner(build_cfg(sae_cfg, training_tokens))
    sae = runner.run()
    metrics, _ = run_evals(  # (all_metrics, feature_metrics); all_metrics is nested
        sae=sae,
        activation_store=runner.activations_store,
        model=runner.model,
        activation_scaler=ActivationScaler(),  # no activation normalization configured
        eval_config=EvalConfig(
            n_eval_reconstruction_batches=n_eval_batches,
            n_eval_sparsity_variance_batches=1,
            compute_sparsity_metrics=True,  # required for evals.py's "l0"
            compute_variance_metrics=True,  # required for "explained_variance"
        ),
        verbose=False,
    )
    return {
        "explained_variance": float(metrics["reconstruction_quality"]["explained_variance"]),
        "l0_evals": float(metrics["sparsity"]["l0"]),
        **heldout_sign_stats(sae, runner.activations_store, sign_batches),
    }

In [11]:
@torch.no_grad()
def heldout_sign_stats(sae, store, n_batches):
    """Sign-sensitive activation stats on held-out batches, counted with != 0."""
    neg = 0.0
    nonzero = 0.0
    tokens = 0
    fired = torch.zeros(sae.cfg.d_sae, dtype=torch.bool)
    for _ in range(n_batches):
        sae_in = store.next_batch()
        acts = sae.encode(sae_in.reshape(-1, sae_in.shape[-1]))
        fired |= (acts != 0).any(dim=0)
        neg += (acts < 0).sum().item()
        nonzero += (acts != 0).sum().item()
        tokens += acts.shape[0]
    return {
        "negative_latent_share": neg / nonzero,
        "true_nonzero_per_token": nonzero / tokens,
        "dead_latent_frac": 1.0 - fired.float().mean().item(),
    }

In [12]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default=None)
    parser.add_argument("--ref", default=None)
    parser.add_argument("--seed", type=int, default=None)
    parser.parse_known_args()  # accepted and ignored

    smoke = os.environ.get("REMYX_SMOKE") == "1"
    training_tokens = 4096 if smoke else 2_048_000  # 1 step vs 500 x 4096
    n_eval_batches = 1 if smoke else 10  # run_evals default batch counts
    sign_batches = 1 if smoke else 3

    print(f"[eval] training TopK control arm ({training_tokens} tokens)", flush=True)
    topk = run_arm(
        TopKTrainingSAEConfig(k=K, d_in=D_IN, d_sae=D_SAE),
        training_tokens,
        n_eval_batches,
        sign_batches,
    )
    if ABSTOPK_AVAILABLE:
        print("[eval] training AbsTopK test arm", flush=True)
        abstopk = run_arm(
            AbsTopKTrainingSAEConfig(k=K, d_in=D_IN, d_sae=D_SAE),
            training_tokens,
            n_eval_batches,
            sign_batches,
        )
    else:
        # Capability absent (pre-change code): degraded zeros for the test arm.
        # The gap then equals TopK's EV and the guardrail reads 0, so the
        # thresholds correctly fail rather than emitting a stand-in number.
        abstopk = {
            "explained_variance": 0.0,
            "l0_evals": 0.0,
            "negative_latent_share": 0.0,
            "true_nonzero_per_token": 0.0,
            "dead_latent_frac": 1.0,
        }

    metrics = {
        "explained_variance_gap__topk_minus_abstopk": topk["explained_variance"]
        - abstopk["explained_variance"],
        "abstopk_negative_latent_share": abstopk["negative_latent_share"],
        "topk_explained_variance": topk["explained_variance"],
        "abstopk_explained_variance": abstopk["explained_variance"],
        "topk_l0_evals": topk["l0_evals"],
        "abstopk_l0_evals": abstopk["l0_evals"],
        "topk_true_nonzero_per_token": topk["true_nonzero_per_token"],
        "abstopk_true_nonzero_per_token": abstopk["true_nonzero_per_token"],
        "topk_negative_latent_share": topk["negative_latent_share"],
        "topk_dead_latent_frac": topk["dead_latent_frac"],
        "abstopk_dead_latent_frac": abstopk["dead_latent_frac"],
        "abstopk_available": int(ABSTOPK_AVAILABLE),
    }
    print(
        json.dumps(
            {k: (round(v, 6) if isinstance(v, float) else v) for k, v in metrics.items()}
        )
    )

if __name__ == "__main__":
    main()

[eval] training TopK control arm (2048000 tokens)


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'roneneldan/TinyStories' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loaded pretrained model tiny-stories-1M into HookedTransformer


/workspace/target_repo/sae_lens/training/activations_store.py:465: UserWarning: Dataset is not tokenized. Pre-tokenizing will improve performance and allows for more control over special tokens. See https://decoderesearch.github.io/SAELens/training_saes/#pretokenizing-datasets for more info.
  warnings.warn(
[remyx] wandb: init -> smellslikeml/remyx-validate-saelens id=a387fce9-feature-s0 group=a387fce9-e50a-4a57-90b6-deedef5a14fe


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


...

wandb: uploading wandb-summary.json; uploading config.yaml


wandb: 
wandb: Run history:
wandb:         details/current_learning_rate ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:            details/n_training_samples ▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
wandb:  losses/auxiliary_reconstruction_loss ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:                       losses/mse_loss █▅▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:                   losses/overall_loss █▅▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:            metrics/explained_variance ▁▄▆▇▇▇▇█████████████████████████████████
wandb:     metrics/explained_variance_legacy ▁▄▆▇▇▇██████████████████████████████████
wandb: metrics/explained_variance_legacy_std █▄▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:                            metrics/l0 ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:                sparsity/dead_features ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:                                    +1 ...
wandb: 
wandb: Run summary:
wandb:         details/current_lear

wandb: 🚀 View run abstopk-1024-LR-0.0003-Tokens-2.048e+06 at: https://forge.coreweave.com/wandb/smellslikeml/remyx-validate-saelens/runs/a387fce9-feature-s0-2
wandb: ⭐️ View project at: https://forge.coreweave.com/wandb/smellslikeml/remyx-validate-saelens
wandb: Synced 4 W&B file(s), 0 media file(s), 5 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261028_063509-a387fce9-feature-s0-2/logs


{"explained_variance_gap__topk_minus_abstopk": -0.00115, "abstopk_negative_latent_share": 0.492408, "topk_explained_variance": 0.966586, "abstopk_explained_variance": 0.967736, "topk_l0_evals": 100.0, "abstopk_l0_evals": 50.877403, "topk_true_nonzero_per_token": 100.0, "abstopk_true_nonzero_per_token": 100.0, "topk_negative_latent_share": 0.0, "topk_dead_latent_frac": 0.447266, "abstopk_dead_latent_frac": 0.366211, "abstopk_available": 1}


## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "Does AbsTopK recover negative signals without hurting reconstruction? Train one TopKTrainingSAE and one AbsTopKTrainingSAE on tiny-stories-1M activations with the same k, width, seed and tokens under 30 minutes on CPU, and compare run_evals explained_variance and L0 side by side."
loop:
  max_iterations: 8
  fix_code: true
benchmarks:
  - name: abstopk-vs-topk-tinystories
    suite: "eval/eval_abstopk_vs_topk.py"
    baseline: none
    scorer: explained_variance_gap__topk_minus_abstopk
    policy:
      guardrail_veto: true
    packages: [wandb]
    metrics:
      - name: explained_variance_gap__topk_minus_abstopk
        direction: min
        threshold: 0.02   # user band: EV_topk - EV_abstopk <= 0.02; typical EV at this scale/tokens is 0.3-0.9, so 0.02 is a tight parity band, reachable by a correct implementation
        role: target
        bar: goal
        reads_as: "How much of TopK's held-out explained_variance AbsTopK fails to recover; 0 or negative means parity or better."
      - name: abstopk_negative_latent_share
        direction: max
        threshold: 0.01   # floor = 1 negative latent of k=100 per token on average = 1/k; ~0.5 expected at symmetric encoder init, so a working signed top-k clears it easily
        role: guardrail
        bar: goal
        reads_as: "Share of AbsTopK's k selected latents that are negative on held-out tokens; TopK's is 0 by construction."
      - name: topk_explained_variance
        direction: max
        threshold: 0.2    # sanity floor: k=100 spans d_in=64, so a trained control sits in the 0.3-0.9 EV band cited above; 0.2 fails only a broken/never-trained control, without which the 0.02 gap target is meaningless
        role: diagnostic
        bar: sanity
        reads_as: "Held-out explained_variance of the TopK control from run_evals; the absolute level the gap is measured against."
      - name: abstopk_explained_variance
        direction: max
        threshold: 0.18   # weakest EV the 0.02 parity band can certify given the control floor: 0.2 - 0.02 = 0.18; reads the sentinel 0.0 when the capability is absent, which must fail
        role: diagnostic
        bar: sanity
        reads_as: "Held-out explained_variance of the AbsTopK arm under the same run_evals protocol."
      - name: topk_l0_evals
        direction: max
        threshold: 95.0   # evals.py counts acts > 0 and TopK keeps the k=100 largest ReLU'd pre-acts out of 1024, so its >0 L0 reads ~100/token; 5 latents of slack tolerate rare exact-zero selections, a broken encode reads far lower
        role: diagnostic
        bar: sanity
        reads_as: "L0 exactly as the repo's evals.py reports it for TopK (counts activations > 0)."
      - name: abstopk_l0_evals
        direction: max
        threshold: 10.0   # evals.py's > 0 count sees only the positive half of the k selections; at the ~0.5 negative share expected at symmetric init this reads ~k/2 = 50, and 10 fails only if essentially no positive latent ever fires
        role: diagnostic
        bar: sanity
        reads_as: "L0 exactly as the repo's evals.py reports it for AbsTopK; its > 0 count cannot see negative latents."
      - name: topk_true_nonzero_per_token
        direction: max
        threshold: 95.0   # counted with != 0: exactly k=100 latents are selected per token and ReLU zeroes only measure-zero ties, so ~100/token; same 5-latent slack as the L0 sanity bar, a broken encode reads far lower
        role: diagnostic
        bar: sanity
        reads_as: "True mean nonzero features per token for TopK, counted with != 0."
      - name: abstopk_true_nonzero_per_token
        direction: max
        threshold: 100   # sanity: signed top-k fires exactly k=100 latents per token (a selected value is 0 only at measure-zero ties); clearing it is not news, missing it means a broken pipeline
        role: diagnostic
        bar: sanity
        reads_as: "True mean nonzero features per token for AbsTopK, counted with != 0; must equal k = 100."
      - name: topk_negative_latent_share
        direction: max
        threshold: 0.0    # exactness bound in the spirit of coverage >= 1.0: ReLU clamps every TopK activation to >= 0, so the correct value is exactly 0 and any negative latent in the control arm is a real bug
        role: diagnostic
        bar: sanity
        reads_as: "Sanity contrast: TopK's negative-latent share; ReLU forces exactly 0."
      - name: topk_dead_latent_frac
        direction: min
        threshold: 0.75   # 3 held-out batches ~ 12,288 tokens x 100 firings = ~1.2M firings over d_sae=1024; a dictionary good enough for the 0.2 EV floor fires well over 256 (> 1024/4) distinct latents, so 0.75 fails only on collapse to a few directions
        role: diagnostic
        bar: sanity
        reads_as: "Fraction of the 1024 latents never active across held-out batches (TopK arm)."
      - name: abstopk_dead_latent_frac
        direction: min
        threshold: 0.75   # same ~1.2M-firing accounting as the control, and signed top-k can use both polarities of each direction so it should match or beat TopK's dead fraction; reads the sentinel 1.0 when the capability is absent, which must fail
        role: diagnostic
        bar: sanity
        reads_as: "Fraction of latents never active for AbsTopK; the aux-loss dead-neuron selection also differs, so this attributes any gap."
compute:
  tier: cpu
  timeout_s: 2700   # one job trains BOTH arms: 2 x 500 steps x ~1-1.5 s/step CPU (tiny-stories-1M + 132k-param SAE) ≈ 17-25 min + tiny-stories-1M/TinyStories download; user cap is 30 min for both arms
requires: [wandb]
held_constant:
  - "model tiny-stories-1M, hook_name blocks.1.hook_mlp_out (d_in=64), device cpu"
  - "k=100 (repo TopK default), d_sae=1024 (16x d_in), identical hyperparameters except the activation"
  - "seed 42 and training_tokens=2,048,000 (500 steps x 4096 train_batch_size_tokens) per arm; a fresh runner per arm so the ActivationsStore draws the same seeded TinyStories stream"
  - "run_evals defaults: n_eval_reconstruction_batches=10, n_eval_sparsity_variance_batches=1"
  - "logger=LoggingConfig(log_to_wandb=True) on both arms; no l1 coefficient (sparsity is structural)"
avoid:
  - "no synthetic or stand-in activations; both arms train on the repo's ActivationsStore streaming TinyStories through tiny-stories-1M"
  - "no torch.cuda calls or GPU device; CPU only per the request"
  - "no wall-clock timing metrics; runtime is not the instrument"
  - "do not count active latents with > 0 in the true-nonzero and negative-share metrics; that is exactly the evals.py definition being contrasted"
  - "no wandb entity or project written by the script; the run environment supplies them"
  - "do not swallow training or eval exceptions; only REMYX_SMOKE=1 may shrink the workload (training_tokens=4096, 1 eval batch, same keys)"
provenance:
  explained_variance_gap__topk_minus_abstopk: "user_guidance (0.02 parity band on run_evals explained_variance)"
  abstopk_negative_latent_share: "user_guidance (nonzero share of selected latents negative); 1/k floor from claim_analysis"
  topk_l0_evals: "user_guidance (the L0 the repo's evals.py reports)"
  abstopk_l0_evals: "user_guidance (the L0 the repo's evals.py reports)"
  topk_true_nonzero_per_token: "user_guidance + PR docs snippet ((acts != 0) count)"
  abstopk_true_nonzero_per_token: "user_guidance + PR docs snippet ((acts != 0) count, equals k)"
  topk_explained_variance: "claim_analysis protocol (run_evals reconstruction_quality.explained_variance)"
  abstopk_explained_variance: "claim_analysis protocol (run_evals reconstruction_quality.explained_variance)"
  topk_negative_latent_share: "inferred sanity contrast (ReLU forces zero)"
  topk_dead_latent_frac: "claim_analysis mismatches (print dead-neuron counts per arm)"
  abstopk_dead_latent_frac: "claim_analysis mismatches (aux-loss selection differs)"
  held_constant: "claim_analysis protocol (k=100 repo default, d_sae=16x d_in, 2,048,000 tokens sized to the 30-minute two-arm CPU budget)"
  suite: "synthesized (repo benchmark dir has no runner for paired SAE training)"
  requires: "repo PR template (Please links to wandb dashboards with a control and test group.)"
  compute: "user_guidance (CPU, under 30 minutes)"
report:
  headline: "AbsTopK trains stably beside a same-seed TopK control on tiny-stories-1M; the verdict is the parity gap plus recovered negative latents, with evals.py L0 undercounting them by construction."
  findings:
    - "Held-out explained_variance: TopK {topk_explained_variance:.3f} vs AbsTopK {abstopk_explained_variance:.3f}; parity gap {explained_variance_gap__topk_minus_abstopk:+.4f} (user band <= 0.02)."
    - "{abstopk_negative_latent_share:%} of AbsTopK's selected latents are negative on held-out tokens, vs {topk_negative_latent_share:%} for TopK (ReLU forces the latter to zero)."
    - "True nonzero count per token counted with != 0: TopK {topk_true_nonzero_per_token:.1f}, AbsTopK {abstopk_true_nonzero_per_token:.1f} (expected = k = 100)."
    - "L0 as evals.py reports it (> 0): TopK {topk_l0_evals:.1f} vs AbsTopK {abstopk_l0_evals:.1f}; the shortfall from 100 is the negative latents that definition cannot see."
    - "Dead-latent fraction across held-out batches: TopK {topk_dead_latent_frac:%} vs AbsTopK {abstopk_dead_latent_frac:%} (the aux-loss selection differs between arms)."
  establishes:
    - "At matched k=100, d_sae=1024, seed and 2.05M tokens on tiny-stories-1M mlp_out, AbsTopK's held-out reconstruction stays within {explained_variance_gap__topk_minus_abstopk} of the same-seed TopK control, resting on run_evals explained_variance."
    - "The signed top-k mechanism operates end-to-end in real training, not just unit tests: {abstopk_negative_latent_share:%} of selected latents are negative and {abstopk_true_nonzero_per_token:.1f} features fire per token."
  does_not_establish:
    - "The paper's four-LLM fidelity improvements or its probing/steering results; this is one 2-layer 64-dim model at ~2M tokens per arm."
    - "Behaviour at production training scale or other k and widths; 'not worse by 0.02' here is a small-scale parity check, not the paper's improvement claim."
    - "That any individual negative feature encodes a bidirectional concept (no probing is run)."
  not_measured:
    - "probing accuracy on bidirectional concepts"
    - "steering success rates and the Difference-in-Mean comparison"
    - "CE-loss / KL knock-on effects (run_evals compute flags left at defaults)"
    - "training throughput or wall-clock on shared CPU"
  caveat: "evals.py counts active latents with > 0, so its L0 structurally undercounts AbsTopK's negative latents; the true-nonzero side-by-side exists to expose that definition gap, not to report extra sparsity."
  next: "Retrain the pair at 100M+ tokens on GPU and add the paper's probing/steering suite to test whether the recovered negative latents encode contrasting concepts."
logging:
  wandb:
    entity: smellslikeml    # from your W&B connection
    project: remyx-validate-saelens    # default
```